

Resource Builder Notebook
---

* Import Required Packages

In [ ]:
import warnings
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

import RES.RESources as RES_module
import RES.visual_styles as styles
from RES import utility as utils
from RES.hdf5_handler import DataHandler

style_path = Path(styles.__file__).parent / "elsevier.mplstyle"
plt.style.use(style_path)
# Suppress specific warnings
warnings.filterwarnings("ignore", category=UserWarning)
cfg=utils.load_config('config/config_WB6.yaml')

## Run the Complete Workflow

----

- All steps are integrated in this 'RES_module.build()' method.

In [ ]:
# Iterate over provinces for both solar and wind resources
resource_types = ['wind','solar']  # 'solar'
countries=['AL','BA','XK','ME','RS']  #'AL','BA','XK','ME','MK','RS'
for country_code in countries:
    for resource_type in resource_types:
        required_args = {
            "config_file_path": 'config/config_WB6.yaml',
            "region_short_code": country_code,
            "resource_type": resource_type
        }
        
        # Create an instance of Resources and execute the module
        Builder = RES_module.RESources_builder(**required_args)
        
    
    # For complete workflow, uncomment the following line
        Builder.build(select_top_sites=False,
                         use_pypsa_buses=False,
                         get_clusters=False)

  └> RES.lands|✓ Stepwise Availability Plots saved to vis/WesternBalkanRegions/AL/default/wind/lands 
  └> RES.lands| ⏳ Plotting explicit impact of layers to Excluder for Albania. This may take a while to compute and plot...
⚠️  RES.lands|'disregard_other_layers' set to TRUE. This parameter should be used exclusively for plotting purposes to showcase land availability impact for individual layers
ℹ️  RES.lands| The Stepwise Land-availability plots and numbers are sensitive to the sequence of layers are loaded. However, the collective impact of layers on final availability is same
ℹ️  RES.lands| The order of loading raster layers mimics the given order in config file under keys: 'GAEZ' and then 'raster_types'
  └─> RES.lands| Loading raster layer 1 'exclusion_areas' to ExclusionContainer ...
  └─> RES.lands| Loading raster layer 2 'terrain_resources' to ExclusionContainer ...
  └─> RES.lands| Loading raster layer 3 'CORINE_land_cover' to ExclusionContainer ...
ℹ️  RES.lands| The order o

## Stepwise Debugging
---

### Define Region/Country of Interest

#### TEMP- CLC

In [ ]:
# CLC_zip_file_path=Path('data/downloaded_data/CORINE/CLC_2028.zip')

# if not CLC_zip_file_path.exists():
#     utils.download_data("https://copernicus-fme.eea.europa.eu/fmedatadownload/results/13521.zip", 'data/downloaded_data/CORINE/CLC_2028.zip')
# raster_file_path=Path('data/downloaded_data/CORINE/u2018_clc2018_v2020_20u1_raster100m/u2018_clc2018_v2020_20u1_raster100m/DATA/U2018_CLC2018_V2020_20u1.tif')
# utils.extract_from_zip('data/downloaded_data/CORINE/CLC_2028.zip', 'data/downloaded_data/CORINE/')
# utils.extract_from_zip('data/downloaded_data/CORINE/extracted/Results/u2018_clc2018_v2020_20u1_raster100m.zip', 'data/downloaded_data/CORINE/u2018_clc2018_v2020_20u1_raster100m')

In [ ]:
# import rasterio

# raster_path = 'data/downloaded_data/CORINE/U2018_CLC2018_V2020_20u1.tif'
# with rasterio.open(raster_path) as src:
#     raster_data = src.read(1)  # Read the first band
#     raster_meta = src.meta

In [ ]:
# # Get total bounds from boundary GeoDataFrame
# minx, miny, maxx, maxy = boundary.total_bounds

# # Create bounding_box_dict with correct keys for downstream use
# bounding_box_dict = {
#     "minx": float(minx),
#     "miny": float(miny),
#     "maxx": float(maxx),
#     "maxy": float(maxy)
# }


In [ ]:
# import rioxarray as rxr

# raster_path = 'data/downloaded_data/CORINE/U2018_CLC2018_V2020_20u1.tif'
# CLC_raster_data = (
#         rxr.open_rasterio(raster_path)
#         .rio.clip_box(**bounding_box_dict)
#         .rename('CF_IEC3')
#         .isel(band=1 if '*Class*' in 'CF_IEC3' else 0)  # 'IEC_Class_ExLoads' data is in band 1
#         .drop_vars(['band', 'spatial_ref'])
#     )


In [ ]:
# Construct region_options as a list of tuples: (name, code)
region_options = [(cfg['region_mapping'][code]['name'], code) for code in cfg['region_mapping']]
region_code = 'BA'  # Default selection, change as needed

# Create dropdown widget for region codes with names shown, codes as values
region_code_dropdown = widgets.Dropdown(
    options=region_options,
    value=region_code,
    description='Region:',
    disabled=False,
)

display(region_code_dropdown)


### Define Resource Type

In [ ]:
resource_type_dropdown = widgets.Dropdown(
    options=['wind', 'solar'],
    value='wind',
    description='Resource:',)
display(resource_type_dropdown)


* Load the modules with set definitions

In [ ]:
resource_type = resource_type_dropdown.value
province_code = region_code_dropdown.value

required_args = {
    "config_file_path": 'config/config_WB6.yaml',
    "region_short_code": province_code,
    "resource_type": resource_type
}


# Create an instance of Resources and execute the module
Builder = RES_module.RESources_builder(**required_args)

- Cleanup store (for fresh results)

In [ ]:
# Builder.clean_data_store()

### Stepwise Checks

#### Step 1: Prepare Spatial Grid Cells

- This method collects the sub-national administrative boundaries. 
- Using that boundary, we calculate the Minimum Bounding Rectangle (MBR). 
- We use that MBR as a cutout to source weather resources data from ERA5 via CDSAPI. The ERA5's cutout is then stored as a netcdf `.nc' file.
- We load that cutout as `atlite`'s `cutout` object.
- We then use `atlite`'s `cutout.grid` attribute to create our test beds for the analysis i.e. the grid cells (geodataframe)

In [ ]:
step1_results=Builder.get_grid_cells()

step1_results.head(5) # See the first 5 rows of the grid cells data

In [ ]:
Builder.datahandler.refresh()
Builder.datahandler.from_store('cells').head(5)

#### Step 2: Calculate Potential Capacity

- This method loads the cutout (atlite's cutout object), regional boundary (GeoDataFrame), loads the cost parameters and  also initiates a __composite excluder__
  - The ([`atlite`'s exclusion container](https://atlite.readthedocs.io/en/master/ref_api.html#atlite.Cutout.availabilitymatrix)) to merge all the spatial layers.
- the `cutout.availabilitymatrix` method calculates % of usable area within each grid cell after applying exclusion criteria (e.g., protected areas, water bodies) and returns an [`AvaliabilityMatrix`](https://atlite.readthedocs.io/en/master/ref_api.html#atlite.Cutout.availabilitymatrix)
- We apply technology landuse intensity (e.g., MW/km² for wind or solar) to translate this to potential capacity data.
- We get the maximum installable capacity for each grid cell based on available area, land use constraints, and technology-specific parameters.
  > - Current results gives a percentage of availability for each grid cell. It does not tell specifically which spatial area inside a grid cell is unavailable.
  > - The _potential capacity_ translation processing involves `area` calculation. The area calculation method is integrated to `RES.cell_processor.get_capacity()`. That method is sensitive to area calculation specific coordinate-system projection of the geodataframe. It is recommended to be cautious about choosing this crs.
  

In [ ]:
step2_results=Builder.get_cell_capacity()

In [ ]:
Builder.cell_processor.capacity_matrix.to_dataframe().reset_index()

In [ ]:
step2_results[0].head(5)

In [ ]:
Builder.datahandler.refresh()
Builder.datahandler.from_store('cells').head(5)

- The Landavailability Map size may need some adjustments to look nicer

In [ ]:
figure=Builder.cell_processor.plot_ERAF5_grid_land_availability(region_boundary=Builder.gadmBoundary.get_region_boundary(),
                                                            Availability_matrix=Builder.cell_processor.Availability_matrix,
                                                            figsize=(9, 8),
                                                            legend_box_x_y=(1.1, 0.9))


#### Step 3: Get CF and Windspeed from Higher Resolution Data

> - Currently configured for Wind Resources only. Wind resources (windspeed) are known to have significant variations across ERA5's ~30km resolution. We rescaled the windspeed with higher resolution windspeed from Global Wind Atlas (GWA). Then we calculate the ERA5 scaled windspeed from the mapped GWA cells. However, GWA does not provide hourly profiles. We source the profile from ERA5.

- returns NONE if result datafield ('windspeed_ERA5') is already there 

In [ ]:
gwa_cells= Builder.gwa_cells.prepare_GWA_data()

* Plot Windspeed for 100m resolution cells (not a mandatory step, for cross checking purposes)
  > That's a high resolution data, may take a while to plot. Make sure your machine's cache memory is enough to hold this data.

### GWA Scaled Wind Speed vs ERA5 Windspeed Comparison

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(9, 5))
sns.histplot(gwa_cells['windspeed_gwa'], bins=50, kde=True, color='steelblue', edgecolor='white', stat='density', alpha=0.6,legend=True)
sns.rugplot(gwa_cells['windspeed_gwa'], color='grey', height=0.02)

plt.title('Windspeed Distribution (GWA\'s 100m Resolution)', fontsize=14)
plt.xlabel('Windspeed (m/s)', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f'vis/{region_code}/gwa_resolution_windspeed_distribution_{region_code}.jpg', dpi=300)
# plt.show()

In [ ]:
step3_results_A=Builder.extract_weather_data()

In [ ]:
step3_results_A

In [ ]:
import geopandas as gpd
# Calculate yearly mean windspeed
wnd_ymean_df = Builder.cutout.data.wnd100m.groupby('time.year').mean('time').to_dataframe(name='windspeed_ERA5').reset_index()

# Create a GeoDataFrame for spatial join
wnd_ymean_gdf = gpd.GeoDataFrame(wnd_ymean_df, geometry=gpd.points_from_xy(wnd_ymean_df['x'], wnd_ymean_df['y']))
wnd_ymean_gdf.crs = step2_results[0].crs

In [ ]:
cells_x=utils.assign_cell_id(step2_results[0],Builder.sub_national_unit_tag)

In [ ]:
import RES.windspeed as wind
X=wind.impute_ERA5_windspeed_to_Cells(Builder.cutout,step2_results[0])

In [ ]:
X.plot()

In [ ]:
step3_results_B=Builder.update_gwa_scaled_params() # testing, 2025 04 21

* A comparison of ERA5 actual (from reanalysis dataset) windspeed vs GWA windspeed downscaled to ERA5 resolution.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

# Plot KDEs
step3_results_B['windspeed_ERA5'].plot.kde(color='orangered', linewidth=2, label='ERA5 Windspeed KDE')
step3_results_B['windspeed_gwa'].plot.kde(color='navy', linewidth=2, label='GWA Windspeed KDE')

# Plot histograms
step3_results_B['windspeed_ERA5'].plot.hist(bins=30, color='orange', edgecolor='white', density=True, alpha=0.4, label='ERA5 Windspeed')
step3_results_B['windspeed_gwa'].plot.hist(bins=30, color='skyblue', edgecolor='white', density=True, alpha=0.4, label='GWA Windspeed')

plt.title('Distribution of GWA and ERA5 Windspeed', fontsize=14, weight='bold')
plt.xlabel('Windspeed (m/s)', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.legend(loc='upper right', frameon=False, fontsize=10)
plt.tight_layout()
plt.savefig(f'vis/{region_code}/ERA5_resolution_windspeed_distribution_ERA5vsGWA_{region_code}.png', dpi=300)
plt.show()

### Step 4: Get Timeseries

- We define technology attributes.
- We extract timeseries using weather resources data from ERA5's cutout.
  - The timeseries calculation method currently configured with [atlite.cutout.pv](https://atlite.readthedocs.io/en/master/ref_api.html#atlite.Cutout.pv) and [atlite.cutout.wind](https://atlite.readthedocs.io/en/master/ref_api.html#atlite.Cutout.wind) methods.

> __Attention__
  > - Configure the timezone conversion information carefully to ensure proper usage of the timeseries in downstream modelling. 
  > - ERA5 provides naive timezone index data. We use the timezone information from config file to enable the timezone shift of the timeseries.
  > - However, after conversion we removed the timezone awareness from the datetime index to harmonize with pypsa supported timeseries index.

In [ ]:
step4_results=Builder.get_CF_timeseries()

### Step 5: Find Grid Proximity

This information is critical for downstream operational analysis with this resource options.

> - Currently configured for Transmission Lines and/or Grid Substations.
> - We do not know the specific project point of a resource. Hence, the resource to grid-node distance has been calculated from the centroid of each grid to the grid node. 
> - If you have a specific project point, you should recalculate this distance with your specific project point.


- Identifies and assigns grid nodes to each cell. 
- Calculates distance (in km) from each grid cell to the nearest grid node (e.g., transmission line, substation) to assess connectivity and feasibility for energy transport.
    
    > If your use case of the resource options are to be plugged in to a downstream operational model (e.g. PyPSA), use harmonized nodes to populate this data.
    > harmonized nodes i.e. same data that are intended to be used as _bus_ nodes at your operational model. 


In [ ]:
step5_results=Builder.find_grid_nodes(use_pypsa_buses=False)  # use_pypsa_buses=False

In [ ]:
step5_results.plot()

### Step 6: Scoring Metric to Rank the Sites

- Scores each grid cell based on multiple criteria (e.g., resource quality, proximity to grid), supporting site selection.

In [ ]:
step6_results=Builder.score_cells()

In [ ]:
step6_results.plot()

In [ ]:
# step6_results.sort_values(by='lcoe_wind').head(10)

In [ ]:
step6_results_filtered=step6_results[step6_results['potential_capacity_wind']>0].sort_values(by='lcoe_wind').head(10)

### Step 7: Clusterized Representation of the Sites

- Groups grid cells into clusters based on spatial or resource characteristics to enable aggregated analysis.
- Produces time series data for each cluster, summarizing the resource and capacity factor information at the cluster leve

In [ ]:
step7_results_Clusters=Builder.get_clusters(step6_results_filtered,
                                               score_tolerance=200,
                                               wcss_tolerance=0.5)
step7_results_ClusterTS=Builder.get_cluster_timeseries()

- Creates Units Dictionary

In [ ]:
utils_dict=Builder.units.create_units_dictionary()

# [Exploratory]

### Explore the outputs from Store

In [ ]:
region_code = region_code_dropdown.value
region_name=cfg.get('region_mapping').get(region_code).get('name') # type: ignore
country_name=cfg.get('country','Western Balkan Region') # type: ignore
country_kwd=country_name.replace(' ','')
utils.print_banner(f"Selected Region: {region_name} ({region_code})")

# Define the directory and search pattern
data_store_dir = Path("../data/store/")
search_keyword = f"resources_{country_kwd}_{region_code}_"

# List files containing the search pattern
matching_files = [str(f) for f in data_store_dir.glob(f"*{search_keyword}*") if f.is_file()]

# Extract run IDs from matching_files
run_ids = [f.replace(str(data_store_dir) + '/', '').replace(search_keyword, '').replace(".h5", '') for f in matching_files]



RUN_ID="default"

# Create dropdown widget
run_id_dropdown = widgets.Dropdown(
    options=run_ids,
    value=RUN_ID,
    description='Run ID:',
    disabled=False,
)

display(run_id_dropdown)

In [ ]:
cells=res_store.from_store('cells')
boundary=res_store.from_store('boundary')
solar_clusters=res_store.from_store('clusters/solar')
wind_clusters=res_store.from_store('clusters/wind')
solar_clusters_ts=res_store.from_store('timeseries/clusters/solar')
wind_clusters_ts=res_store.from_store('timeseries/clusters/wind')

In [ ]:
# Initialize an empty list to store the results
results = []

# Iterate through each cluster
for cluster, row in all_clusters.iterrows():
    
    # Extract the cluster's region and cluster number
    region = row[sub_national_unit_tag]
    print(region)
    cluster_no = row['Cluster_No']  # Dynamically fetch the cluster number from the row
    print(cluster_no)
    # Get the cell indices for the cluster based on the region and cluster number
    cluster_cell_indices = dissolved_indices.loc[region][cluster_no]
    print(cluster_cell_indices)
    
    existing_cols = [col for col in cluster_cell_indices if col in cells_timeseries.columns]
    print(existing_cols)
    if not existing_cols:
        print(f"Warning: No valid timeseries columns found for cluster {cluster}")
        continue
    cluster_ts = cells_timeseries[existing_cols].mean(axis=1)

    # Store the mean as a DataFrame with the cluster name as the column name
    results.append(pd.DataFrame(cluster_ts, columns=[cluster]))

- Interactive Map

In [ ]:
# wind_clusters[wind_clusters['lcoe']<=100].explore('potential_capacity')

# Playground for Top Site Selection

In [ ]:
resource_clusters_wind,cluster_timeseries_wind=RES_module.select_top_sites(wind_clusters,
                                                                wind_clusters_ts,
                                                                    resource_max_capacity=50)

In [ ]:
resource_clusters_solar,cluster_timeseries_solar=RES_module.select_top_sites(solar_clusters,
                                                                solar_clusters_ts,
                                                                    resource_max_capacity=10)

resource_clusters_wind,cluster_timeseries_wind=RES_module.select_top_sites(wind_clusters,
                                                                wind_clusters_ts,
                                                                    resource_max_capacity=50)

In [ ]:
RES_module.export_results('wind',
                          country_code,
                    resource_clusters_wind,
                    cluster_timeseries_wind,)

In [ ]:
RES_module.export_results('solar',
                             country_code,
                    resource_clusters_solar,
                    cluster_timeseries_solar,)

In [ ]:
# resource_clusters_solar.plot('potential_capacity',legend=True)
# resource_clusters_wind.plot('potential_capacity',legend=True)

# Visualize Results

In [ ]:
sub_national_unit_tag='City'

In [ ]:
legend_x_ax_offset = 1.4
legend_y_ax_offset = 0.05

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Ensure 'Region' is in the columns for both boundary and cells
if sub_national_unit_tag not in boundary.columns:
    boundary = boundary.reset_index(inplace=True)

# Assign a number to each region
boundary['Region_Number'] = range(1, len(boundary) + 1)

# Define custom bins and labels for solar and wind capacity
solar_bins = [0, 100, 200, 300, 500, float('inf')]  # Custom ranges
solar_labels = ['<100','100-200', '200-300', '300-500','>500']  # Labels for legend

# Define custom bins and labels for solar and wind capacity
wind_bins = [0, 300, 500, 1000, 2000,3000, float('inf')]  # Custom ranges
wind_labels = ['<300','300-500', '500-1000', '1000-2000','2000-3000', '>3000']  # Labels for legend

# Categorize potential_capacity_solar and potential_capacity_wind into bins
resource_clusters_solar['solar_category'] = pd.cut(resource_clusters_solar['potential_capacity'], bins=solar_bins, labels=solar_labels, include_lowest=True)
resource_clusters_wind['wind_category'] = pd.cut(resource_clusters_wind['potential_capacity'], bins=wind_bins, labels=wind_labels, include_lowest=True)

# Create figure and axes for side-by-side plotting
fig, (ax1, ax2) = plt.subplots(figsize=(18, 8), ncols=2)
fig.suptitle("Potential Sites for Targeted Capacity Investments", fontsize=16,weight='bold')
# Set axis off for both subplots
ax1.set_axis_off()
ax2.set_axis_off()

# Shadow effect offset
shadow_offset = 0.005

# Plot solar map on ax1
# Add shadow effect for solar map
boundary.geometry = boundary.geometry.translate(xoff=shadow_offset, yoff=-shadow_offset)
boundary.plot(ax=ax1, color='None', edgecolor='grey', linewidth=1, alpha=0.7)  # Shadow layer
boundary.geometry = boundary.geometry.translate(xoff=-shadow_offset, yoff=shadow_offset)

# Plot solar cells
resource_clusters_solar.plot(column='solar_category', ax=ax1, cmap='Wistia', legend=True, 
           legend_kwds={'title': "Solar Capacity (MW)", 'loc': 'upper right','bbox_to_anchor':(legend_x_ax_offset,1), 'frameon': False})

# Plot actual boundary for solar map
boundary.plot(ax=ax1, facecolor='none', edgecolor='black', linewidth=0.2, alpha=0.7)
"""
# Annotate region numbers for solar map
for idx, row in boundary.iterrows():
    centroid = row.geometry.centroid
    ax1.annotate(f"{row['Region_Number']}", 
                 xy=(centroid.x, centroid.y), 
                 ha='center', va='center',
                 fontsize=7, color='black',
                 bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, boxstyle='round,pad=0.2'))
"""
# Plot wind map on ax2
# Add shadow effect for wind map
boundary.geometry = boundary.geometry.translate(xoff=shadow_offset, yoff=-shadow_offset)
boundary.plot(ax=ax2, color='None', edgecolor='grey', linewidth=1, alpha=0.7)  # Shadow layer
boundary.geometry = boundary.geometry.translate(xoff=-shadow_offset, yoff=shadow_offset)

# Plot wind cells
resource_clusters_wind.plot(column='wind_category', ax=ax2, cmap='summer', legend=True, 
           legend_kwds={'title': "Wind Capacity (MW)", 'bbox_to_anchor':(legend_x_ax_offset,1), 'frameon': False})

# Plot actual boundary for wind map
boundary.plot(ax=ax2, facecolor='none', edgecolor='black', linewidth=0.2, alpha=0.7)
"""
# Annotate region numbers for wind map
for idx, row in boundary.iterrows():
    centroid = row.geometry.centroid
    ax2.annotate(f"{row['Region_Number']}", 
                 xy=(centroid.x, centroid.y), 
                 ha='center', va='center',
                 fontsize=8, color='black',
                 bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, boxstyle='round,pad=0.2'))
"""
# Adjust layout for cleaner appearance
fig.patch.set_alpha(0)  # Make figure background transparent
plt.tight_layout()

# Show the side-by-side plot

# plt.savefig('solar_wind_capacity_map.png',dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Ensure 'Region' column is present
if sub_national_unit_tag not in boundary.columns:
    boundary = boundary.reset_index(inplace=True)

# Assign a number to each region
boundary['Region_Number'] = range(1, len(boundary) + 1)

# Create figure and axes
fig, (ax1, ax2) = plt.subplots(figsize=(12, 8), ncols=2)
fig.suptitle("Potential Sites for Targeted Capacity Investments", fontsize=16, weight='bold')

# Turn off axes
ax1.set_axis_off()
ax2.set_axis_off()

# Shadow offset
shadow_offset = 0.005

# --- Solar map ---
# Shadow
boundary.geometry = boundary.geometry.translate(xoff=shadow_offset, yoff=-shadow_offset)
boundary.plot(ax=ax1, color='None', edgecolor='grey', linewidth=1, alpha=0.7)
boundary.geometry = boundary.geometry.translate(xoff=-shadow_offset, yoff=shadow_offset)

# Continuous colormap for solar
resource_clusters_solar.plot(
    column='potential_capacity', 
    ax=ax1, 
    cmap='Wistia', 
    legend=True,
    legend_kwds={
        'label': "Solar Capacity (MW)",
        'shrink': 0.7
    }
)

boundary.plot(ax=ax1, facecolor='none', edgecolor='black', linewidth=0.2, alpha=0.7)

# --- Wind map ---
# Shadow
boundary.geometry = boundary.geometry.translate(xoff=shadow_offset, yoff=-shadow_offset)
boundary.plot(ax=ax2, color='None', edgecolor='grey', linewidth=1, alpha=0.7)
boundary.geometry = boundary.geometry.translate(xoff=-shadow_offset, yoff=shadow_offset)

# Continuous colormap for wind
resource_clusters_wind.plot(
    column='potential_capacity', 
    ax=ax2, 
    cmap='summer', 
    legend=True,
    legend_kwds={
        'label': "Wind Capacity (MW)",
        'shrink': 0.7
    }
)

boundary.plot(ax=ax2, facecolor='none', edgecolor='black', linewidth=0.2, alpha=0.7)

# Final touches
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()
